# 01 · Data Cleaning — Healthcare Claims

**Goal:** turn three raw exports (patients, providers, claims) into analysis-ready tables.

Raw data problems this notebook fixes:
1. Duplicate claim rows (double submissions)
2. Mixed date formats (`YYYY-MM-DD` and `MM/DD/YYYY`)
3. Inconsistent categorical coding (`F`, `female`, `Female`; `paid`, `PAID`)
4. Missing and negative billed amounts
5. Stray whitespace and casing in provider specialties
6. Missing patient state

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "Python" else Path.cwd()
RAW, CLEAN = ROOT / "Data" / "raw", ROOT / "Data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)

patients  = pd.read_csv(RAW / "patients.csv")
providers = pd.read_csv(RAW / "providers.csv")
claims    = pd.read_csv(RAW / "claims.csv", dtype={"cpt_code": str})
for name, df in [("patients", patients), ("providers", providers), ("claims", claims)]:
    print(f"{name:<10} {df.shape[0]:>7,} rows  {df.shape[1]} cols")

patients     8,000 rows  6 cols
providers      250 rows  5 cols
claims      50,750 rows  14 cols


## 1. Profile the raw data

In [2]:
def profile(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "null_%": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(),
    })

profile(claims)

,dtype,nulls,null_%,unique
claim_id,object,0,0.00,50000
patient_id,object,0,0.00,7946
provider_id,object,0,0.00,250
service_date,object,0,0.00,1440
submission_date,object,0,0.00,831
processed_date,object,2652,5.23,809
claim_type,object,0,0.00,3
diagnosis_category,object,0,0.00,11
cpt_code,object,0,0.00,18
billed_amount,float64,124,0.24,42519


In [3]:
print("Exact duplicate claim rows:", claims.duplicated().sum())
print("Duplicate claim_ids:       ", claims["claim_id"].duplicated().sum())
print("\nclaim_status values:", claims["claim_status"].value_counts().to_dict())
print("\ngender values:", patients["gender"].value_counts().to_dict())
print("\nNegative billed amounts:", (claims["billed_amount"] < 0).sum())
print("Sample service_date formats:", claims["service_date"].sample(8, random_state=1).tolist())

Exact duplicate claim rows: 750
Duplicate claim_ids:        750

claim_status values: {'Paid': 41649, 'Denied': 5783, 'Pending': 2607, 'PAID': 312, 'paid': 279, 'denied': 42, 'DENIED': 33, 'PENDING': 25, 'pending': 20}

gender values: {'F': 3818, 'M': 3582, ' f': 118, 'Female': 110, 'female': 104, 'Male': 94, 'male': 87, 'M ': 87}

Negative billed amounts: 46
Sample service_date formats: ['05/06/2024', '2025-03-20', '2025-02-17', '2025-06-24', '2024-12-28', '2024-11-02', '2024-03-02', '2024-09-24']


## 2. Patients

In [4]:
patients["gender"] = (patients["gender"].str.strip().str.upper().str[0]
                      .map({"F": "Female", "M": "Male"}))
patients["state"] = patients["state"].fillna("Unknown")
for col in ["date_of_birth", "enrollment_date"]:
    patients[col] = pd.to_datetime(patients[col])

assert patients["gender"].notna().all()
assert patients["patient_id"].is_unique
patients["gender"].value_counts()

gender
Female    4150
Male      3850
Name: count, dtype: int64

## 3. Providers

In [5]:
providers["specialty"] = providers["specialty"].str.strip().str.title()
assert providers["provider_id"].is_unique
providers["specialty"].value_counts()

specialty
Primary Care          59
Behavioral Health     31
Hospital              31
Emergency Medicine    30
Radiology             25
Pediatrics            24
Orthopedics           22
Cardiology            21
Oncology               7
Name: count, dtype: int64

## 4. Claims

Decisions (documented so a reviewer can challenge them):
- **Duplicates:** drop exact duplicate rows; the `claim_id` is then unique.
- **Dates:** parse with `format="mixed"` so both formats are read correctly.
- **Negative billed amounts:** a sign error in the export, so take the absolute value.
- **Missing billed amounts:** impute the median billed amount for the same CPT code, floored at the allowed amount (a first attempt without the floor failed the `paid <= allowed <= billed` check below). A flag column keeps the imputation visible.
- **Denial reason:** `Not Applicable` for claims that were not denied.

In [6]:
before = len(claims)
claims = claims.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(claims):,} duplicate rows")
assert claims["claim_id"].is_unique

for col in ["service_date", "submission_date", "processed_date"]:
    claims[col] = pd.to_datetime(claims[col], format="mixed")

claims["claim_status"] = claims["claim_status"].str.strip().str.title()

neg = claims["billed_amount"] < 0
claims.loc[neg, "billed_amount"] = claims.loc[neg, "billed_amount"].abs()
print(f"Fixed {neg.sum()} negative billed amounts")

claims["billed_imputed"] = claims["billed_amount"].isna()
cpt_median = claims.groupby("cpt_code")["billed_amount"].transform("median")
# a provider never bills less than the payer allowed, so the imputed value is floored at allowed_amount
claims["billed_amount"] = claims["billed_amount"].fillna(
    np.maximum(cpt_median, claims["allowed_amount"])).round(2)
print(f"Imputed {claims['billed_imputed'].sum()} missing billed amounts")

claims["denial_reason"] = claims["denial_reason"].fillna("Not Applicable")
claims["claim_status"].value_counts()

Removed 750 duplicate rows


Fixed 45 negative billed amounts
Imputed 120 missing billed amounts


claim_status
Paid       41611
Denied      5772
Pending     2617
Name: count, dtype: int64

## 5. Feature engineering

In [7]:
claims = claims.merge(patients[["patient_id", "date_of_birth", "plan_type"]], on="patient_id", how="left")
claims = claims.merge(providers[["provider_id", "specialty", "network_status"]], on="provider_id", how="left")

claims["service_month"]   = claims["service_date"].dt.to_period("M").astype(str)
claims["service_year"]    = claims["service_date"].dt.year
claims["age_at_service"]  = ((claims["service_date"] - claims["date_of_birth"]).dt.days // 365.25).astype(int)
claims["age_group"]       = pd.cut(claims["age_at_service"], [-1, 17, 34, 49, 64, 200],
                                   labels=["0-17", "18-34", "35-49", "50-64", "65+"]).astype(str)
claims["days_to_submit"]  = (claims["submission_date"] - claims["service_date"]).dt.days
claims["days_to_process"] = (claims["processed_date"] - claims["submission_date"]).dt.days
claims["is_denied"]       = (claims["claim_status"] == "Denied").astype(int)
claims["member_responsibility"] = np.where(claims["claim_status"] == "Paid",
                                           claims["allowed_amount"] - claims["paid_amount"], 0).round(2)
claims = claims.drop(columns=["date_of_birth"])
claims.head()

,claim_id,patient_id,provider_id,service_date,submission_date,processed_date,claim_type,diagnosis_category,cpt_code,billed_amount,...,specialty,network_status,service_month,service_year,age_at_service,age_group,days_to_submit,days_to_process,is_denied,member_responsibility
0,C027579,P07368,PR0247,2024-05-24,2024-05-27,2024-06-11,Professional,Mental Health,99203,331.78,...,Behavioral Health,In-Network,2024-05,2024,90,65+,3,15.0,0,31.96
1,C019900,P05824,PR0006,2025-06-03,2025-06-15,2025-06-25,Outpatient,Injury,70450,952.77,...,Emergency Medicine,In-Network,2025-06,2025,27,18-34,12,10.0,0,106.08
2,C011373,P03963,PR0061,2025-05-01,2025-05-05,2025-05-25,Professional,Infectious Disease,99204,165.63,...,Pediatrics,In-Network,2025-05,2025,83,65+,4,20.0,0,15.61
3,C038815,P06931,PR0065,2024-07-24,2024-08-09,2024-08-24,Inpatient,Musculoskeletal,99232,7336.58,...,Hospital,In-Network,2024-07,2024,18,18-34,16,15.0,0,1002.18
4,C008756,P02881,PR0030,2024-09-05,2024-09-11,2024-09-21,Professional,Mental Health,99214,267.26,...,Behavioral Health,In-Network,2024-09,2024,10,0-17,6,10.0,1,0.00


## 6. Validation checks

In [8]:
checks = {
    "claim_id unique":                   claims["claim_id"].is_unique,
    "no missing billed amounts":         claims["billed_amount"].notna().all(),
    "no negative money":                 (claims[["billed_amount", "allowed_amount", "paid_amount"]] >= 0).all().all(),
    "paid <= allowed <= billed":         ((claims["paid_amount"] <= claims["allowed_amount"] + 0.01)
                                          & (claims["allowed_amount"] <= claims["billed_amount"] + 0.01)).all(),
    "submission on/after service":       (claims["days_to_submit"] >= 0).all(),
    "only 3 statuses":                   set(claims["claim_status"]) == {"Paid", "Denied", "Pending"},
    "denied claims have a reason":       (claims.loc[claims["is_denied"] == 1, "denial_reason"] != "Not Applicable").all(),
    "pending claims have no processed date": claims.loc[claims["claim_status"] == "Pending", "processed_date"].isna().all(),
    "every claim matches a patient":     claims["plan_type"].notna().all(),
    "every claim matches a provider":    claims["specialty"].notna().all(),
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)
assert all(checks.values())

PASS claim_id unique
PASS no missing billed amounts
PASS no negative money
PASS paid <= allowed <= billed
PASS submission on/after service
PASS only 3 statuses
PASS denied claims have a reason
PASS pending claims have no processed date
PASS every claim matches a patient
PASS every claim matches a provider


## 7. Save clean tables

In [9]:
patients.to_csv(CLEAN / "patients_clean.csv", index=False, date_format="%Y-%m-%d")
providers.to_csv(CLEAN / "providers_clean.csv", index=False)
claims.to_csv(CLEAN / "claims_clean.csv", index=False, date_format="%Y-%m-%d")
print(f"Saved {len(claims):,} claims, {len(patients):,} patients, {len(providers):,} providers to {CLEAN.relative_to(ROOT)}")

Saved 50,000 claims, 8,000 patients, 250 providers to Data\clean
